# Task 3 - Ultra Linear Models (The Best For Sparse Text)

Tree models (like LightGBM) struggle heavily on sparse distributions with massive dimensions.
Linear models thrive on it! 

Here we expand our vocabulary space to **150,000** unigrams, bigrams, and trigrams.
We fit three exceptionally strong linear algorithms:
1. **LogisticRegression** (Fast `lbfgs` implementation with L2 penalty)
2. **LinearSVC** (L2 Penalty)
3. **RidgeClassifier**

Finally, a Hard Voting Ensemble lets all three linear boundaries "vote" to correct each other's individual mistakes!

In [1]:
# Section 0 - Imports & Config
import os
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import VotingClassifier

warnings.filterwarnings('ignore')

CONFIG = {
    'TRAIN_PATH': 'dataset/train.csv',
    'TEST_PATH': 'dataset/test.csv',
    'SUBMISSIONS_DIR': 'submissions',
    'RANDOM_STATE': 42,
    'VAL_SIZE': 0.15,
    # We go massive: 150k max features + Unigrams/Bigrams/Trigrams
    'TFIDF': {
        'max_features': 150000,
        'ngram_range': (1, 3),
        'min_df': 2,
        'max_df': 0.90,
        'sublinear_tf': True,
    }
}

os.makedirs(CONFIG['SUBMISSIONS_DIR'], exist_ok=True)
np.random.seed(CONFIG['RANDOM_STATE'])
print('Config loaded.')

Config loaded.


## Section 1 - Load Data & Split

In [2]:
if not os.path.exists(CONFIG['TRAIN_PATH']):
    raise FileNotFoundError("Please place train.csv and test.csv in the dataset/ directory!")

train_df = pd.read_csv(CONFIG['TRAIN_PATH'], sep='\t')
test_df = pd.read_csv(CONFIG['TEST_PATH'], sep='\t')

train_texts = train_df['abstract'].astype(str)
train_labels = train_df['label_id'].astype(int)
test_texts = test_df['abstract'].astype(str)
test_ids = test_df['id'].astype(int)

# Validation Split
X_train_text, X_val_text, y_train, y_val = train_test_split(
    train_texts, train_labels, test_size=CONFIG['VAL_SIZE'], 
    random_state=CONFIG['RANDOM_STATE'], stratify=train_labels
)

print(f'Train split: {len(X_train_text)} samples')
print(f'Val split:   {len(X_val_text)} samples')

Train split: 118282 samples
Val split:   20874 samples


## Section 2 - Massive TF-IDF Generation
Expect this to take ~30-60 seconds due to trigrams.

In [3]:
print('Fitting 150k TF-IDF...')
t0 = time.time()
vec = TfidfVectorizer(**CONFIG['TFIDF'])
X_train_vec = vec.fit_transform(X_train_text)
X_val_vec = vec.transform(X_val_text)
X_test_vec = vec.transform(test_texts)
print(f'TF-IDF Done in {time.time()-t0:.2f}s. Shape: {X_train_vec.shape}')

Fitting 150k TF-IDF...
TF-IDF Done in 190.03s. Shape: (118282, 150000)


## Section 3 - Sweeping Top Linear Models
We train them and check their individual validation scores.

In [4]:
# 1. Logistic Regression (C=2.5 often helps slightly over C=1.0 for multi-class)
logreg = LogisticRegression(
    C=2.5, 
    max_iter=1000, 
    solver='lbfgs',
    n_jobs=-1,
    random_state=CONFIG['RANDOM_STATE']
)

# 2. LinearSVC (C=1.0 is historically ideal)
svc = LinearSVC(C=1.0, max_iter=2000, random_state=CONFIG['RANDOM_STATE'])

# 3. RidgeClassifier (L2 norm on purely squared error)
ridge = RidgeClassifier(alpha=1.0, random_state=CONFIG['RANDOM_STATE'])

models = {
    'LogisticRegression (C=2.5)': logreg,
    'LinearSVC (C=1.0)': svc,
    'RidgeClassifier (alpha=1.0)': ridge
}

for name, model in models.items():
    print(f'Training {name}...')
    t0 = time.time()
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_val_vec)
    f1 = f1_score(y_val, preds, average='macro')
    print(f"{name} Val Macro F1: {f1:.4f} (Time: {time.time()-t0:.2f}s)")

Training LogisticRegression (C=2.5)...
LogisticRegression (C=2.5) Val Macro F1: 0.6314 (Time: 518.16s)
Training LinearSVC (C=1.0)...
LinearSVC (C=1.0) Val Macro F1: 0.6541 (Time: 35.83s)
Training RidgeClassifier (alpha=1.0)...
RidgeClassifier (alpha=1.0) Val Macro F1: 0.6297 (Time: 120.92s)


## Section 4 - The Hard Voting Linear Ensemble

In [5]:
print('Training Hard Voting Ensemble...')
t0 = time.time()
ensemble = VotingClassifier(
    estimators=[
        ('lr', logreg),
        ('svc', svc),
        ('ridge', ridge)
    ],
    voting='hard',   # take the majority class prediction among the 3 models
    n_jobs=-1
)

ensemble.fit(X_train_vec, y_train)
ens_preds = ensemble.predict(X_val_vec)
ens_f1 = f1_score(y_val, ens_preds, average='macro')
print(f"Ensemble Val Macro F1: {ens_f1:.4f} (Time: {time.time()-t0:.2f}s)")

Training Hard Voting Ensemble...
Ensemble Val Macro F1: 0.6475 (Time: 570.77s)


## Section 5 - Generating Both Final CSVs

In [6]:
print('Generating full Kaggle test predictions...')
# Output the best individual model (often Logistic Regression is wildly strong)
lr_test_preds = logreg.predict(X_test_vec)
lr_sub = pd.DataFrame({'id': test_ids, 'label_id': lr_test_preds})
lr_sub.to_csv(os.path.join(CONFIG['SUBMISSIONS_DIR'], 'task3_logistic_regression.csv'), index=False)

# Output the ensemble model
ens_test_preds = ensemble.predict(X_test_vec)
ens_sub = pd.DataFrame({'id': test_ids, 'label_id': ens_test_preds})
ens_sub.to_csv(os.path.join(CONFIG['SUBMISSIONS_DIR'], 'task3_linear_ensemble.csv'), index=False)

print('Saved task3_logistic_regression.csv and task3_linear_ensemble.csv.')

Generating full Kaggle test predictions...
Saved task3_logistic_regression.csv and task3_linear_ensemble.csv.
